# Data Cleaning - sales_data_raw.xlsx
## Project: Product Placement Optimisation
### Purpose: Fix all issues found in 01_data_audit.ipynb
## Student: Samikshya Baniya
## Student ID: 230360
## Module: ST6001CEM Individual Project
### Input: data/raw/sales_data_raw.xlsx
### Output: data/processed/sales_data_cleaned.csv

## Cleaning Steps Planned

1. Load raw data
2. Rename all columns to clean names
3. Drop DISCOUNT and ROUND OFF columns
4. Strip trailing spaces from Product Group and Product
5. Standardize product group names to 25 clean categories
6. Fix misclassified products
7. Remove 1,040 duplicate rows
8. Remove 1 blank row and 1 missing product group row
9. Save cleaned data to data/processed/sales_data_cleaned.csv

## Step 1: Load Raw Data

In [1]:
import pandas as pd
import numpy as np

df = pd.read_excel(
    r'D:\softwarica\Sem 6\Individual Project\product-placement-optimization\data\raw\sales_data_raw.xlsx',
    header=7
)

print(f"Raw data loaded!")
print(f"Shape: {df.shape}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

Raw data loaded!
Shape: (768222, 14)
Rows: 768,222
Columns: 14


## Step 2: Rename Columns

In [2]:
df = df.rename(columns={
    'Date':           'date',
    'Miti':           'date_nepali',
    'Inv.No':         'invoice_no',
    'Customer':       'customer',
    'Product Group':  'product_group',
    'Product':        'product',
    'Uom':            'unit',
    'Qty':            'quantity',
    'Rate':           'unit_price',
    'B. Amount':      'base_amount',
    'DISCOUNT':       'discount',
    'VAT':            'vat',
    'ROUND OFF':      'round_off',
    'Total Amount':   'total_amount'
})

print("Columns renamed successfully!")
print(df.columns.tolist())

Columns renamed successfully!
['date', 'date_nepali', 'invoice_no', 'customer', 'product_group', 'product', 'unit', 'quantity', 'unit_price', 'base_amount', 'discount', 'vat', 'round_off', 'total_amount']


## Step 3: Drop Unnecessary Columns

In [3]:
df = df.drop(columns=['discount', 'round_off'])

print("Dropped discount and round_off columns!")
print(f"Remaining columns: {df.columns.tolist()}")
print(f"Shape now: {df.shape}")

Dropped discount and round_off columns!
Remaining columns: ['date', 'date_nepali', 'invoice_no', 'customer', 'product_group', 'product', 'unit', 'quantity', 'unit_price', 'base_amount', 'vat', 'total_amount']
Shape now: (768222, 12)


## Step 4: Clean Product Group Column

In [4]:
df['product_group_clean'] = (
    df['product_group']
    .astype(str)
    .str.strip()
    .str.upper()
    .str.replace('_', ' ', regex=False)
    .str.replace(r'\s+', ' ', regex=True)
)

print("Product group cleaned successfully!")
print(f"Original unique groups:  {df['product_group'].nunique()}")
print(f"Cleaned unique groups:   {df['product_group_clean'].nunique()}")

Product group cleaned successfully!
Original unique groups:  38
Cleaned unique groups:   37


In [5]:
print("ALL CLEANED PRODUCT GROUPS (alphabetical):")
print("-" * 60)
for i, group in enumerate(sorted(df['product_group_clean'].dropna().unique()), 1):
    print(f"{i:2d}. '{group}'")

ALL CLEANED PRODUCT GROUPS (alphabetical):
------------------------------------------------------------
 1. 'BABY CARE'
 2. 'BAKERY ITEM'
 3. 'BATTISA MASALA'
 4. 'BIRTHDAY ITEM'
 5. 'BISCUITS'
 6. 'CANNED AND PACK GOODS'
 7. 'CANNED AND PACK ITEM'
 8. 'CEREALS'
 9. 'CHOCOLATE & CANDY'
10. 'CIGARETTE'
11. 'COFFEE ITEMS'
12. 'COOKING OIL'
13. 'DAIRY PRODUCT'
14. 'DRY FRUIT ITEMS'
15. 'ELECTRICAL SUPPLIES'
16. 'ESSENTIAL FOOD ITEM'
17. 'FROZEN'
18. 'GAULE'
19. 'HARD DRINKS'
20. 'HEALTH & WELLNESS'
21. 'HOUSEHOLD ITEM'
22. 'KOREAN ITEMS'
23. 'LAXMI DALMOT'
24. 'MEAT AND SEAFOOD'
25. 'NOODLES ITEMS'
26. 'PAICHO ITEMS'
27. 'PERSONAL CARE'
28. 'PET CARE'
29. 'POOJA ITEMS'
30. 'RICE'
31. 'SOAP AND CLEANER'
32. 'SOFT DRINK'
33. 'STATIONARY ITEM'
34. 'SURTI'
35. 'SWEETS'
36. 'TEA & MASALA ITEM'
37. 'VEGETABLES AND FRUITS'


### Why we clean the product group column
The raw product group names have trailing spaces, inconsistent capitalisation, and underscores. For example "food staples" and "FOOD_STAPLES" are the same thing but Python sees them as different. Stripping spaces, converting to uppercase, and removing underscores collapses 38 messy names into a consistent set ready for mapping.

## Step 5: Create Standardized Category Column

In [6]:
category_mapping = {
    # RENAMES
    'ESSENTIAL FOOD ITEM':   'FOOD STAPLES',
    'CANNED AND PACK GOODS': 'CANNED AND PACKAGED FOODS',
    'SOAP AND CLEANER':      'CLEANING SUPPLIES',
    'BISCUITS':              'BISCUITS AND COOKIES',
    'CHOCOLATE & CANDY':     'CONFECTIONERY',
    'TEA & MASALA ITEM':     'TEA AND SPICES',
    'HOUSEHOLD ITEM':        'HOUSEHOLD ITEMS',
    'NOODLES ITEMS':         'NOODLES',
    'DAIRY PRODUCT':         'DAIRY PRODUCTS',
    'FROZEN':                'FROZEN FOODS',
    'STATIONARY ITEM':       'STATIONERY',
    'BAKERY ITEM':           'BAKERY',
    'CEREALS':               'BREAKFAST CEREALS',
    'LAXMI DALMOT':          'NAMKEEN AND SNACKS',
    'VEGETABLES AND FRUITS': 'FRUITS AND VEGETABLES',
    'HEALTH & WELLNESS':     'HEALTH AND WELLNESS',
    'BIRTHDAY ITEM':         'PARTY SUPPLIES',
    'COFFEE ITEMS':          'TEA AND SPICES',
    'DRY FRUIT ITEMS':       'CANNED AND PACKAGED FOODS',
    'HARD DRINKS':           'ALCOHOLIC BEVERAGES',
    'SOFT DRINK':            'SOFT DRINKS AND JUICES',
    'CIGARETTE':             'CIGARETTE AND TOBACCO',

    # MERGES
    'CANNED AND PACK ITEM':  'CANNED AND PACKAGED FOODS',
    'SWEETS':                'CONFECTIONERY',
    'BATTISA MASALA':        'TEA AND SPICES',
    'MEAT AND SEAFOOD':      'FROZEN FOODS',
    'SURTI':                 'CIGARETTE AND TOBACCO',
    'GAULE':                 'FOOD STAPLES',
    'PAICHO ITEMS':          'CANNED AND PACKAGED FOODS',
    'KOREAN ITEMS':          'NAMKEEN AND SNACKS',
}

df['category'] = df['product_group_clean'].replace(category_mapping)

before = len(df)
df = df[df['category'] != 'PET CARE']
print(f"Dropped PET CARE rows: {before - len(df)}")
print(f"Total rows remaining: {len(df):,}")

Dropped PET CARE rows: 1
Total rows remaining: 768,221


In [7]:
print("FINAL STANDARDIZED CATEGORIES:")
print("-" * 60)
for i, (cat, count) in enumerate(df['category'].value_counts().items(), 1):
    pct = (count / len(df)) * 100
    print(f"{i:2d}. {cat:35s} | {count:,} rows ({pct:.1f}%)")

print(f"\nTotal categories: {df['category'].nunique()}")
print(f"Total rows: {len(df):,}")

FINAL STANDARDIZED CATEGORIES:
------------------------------------------------------------
 1. FOOD STAPLES                        | 178,872 rows (23.3%)
 2. CANNED AND PACKAGED FOODS           | 98,784 rows (12.9%)
 3. CLEANING SUPPLIES                   | 50,868 rows (6.6%)
 4. BISCUITS AND COOKIES                | 47,062 rows (6.1%)
 5. CONFECTIONERY                       | 45,735 rows (6.0%)
 6. TEA AND SPICES                      | 45,520 rows (5.9%)
 7. PERSONAL CARE                       | 43,296 rows (5.6%)
 8. COOKING OIL                         | 37,265 rows (4.9%)
 9. HOUSEHOLD ITEMS                     | 36,091 rows (4.7%)
10. NOODLES                             | 31,964 rows (4.2%)
11. SOFT DRINKS AND JUICES              | 30,832 rows (4.0%)
12. DAIRY PRODUCTS                      | 29,506 rows (3.8%)
13. RICE                                | 16,123 rows (2.1%)
14. CIGARETTE AND TOBACCO               | 14,296 rows (1.9%)
15. POOJA ITEMS                         | 10,663 ro

In [8]:
hw = df[df['category'] == 'HEALTH AND WELLNESS']['product'].value_counts()
print("Products in HEALTH AND WELLNESS:")
print(hw)

Products in HEALTH AND WELLNESS:
product
Strepsils 8p                         528
SugarFree Gold 500Tab                 88
SugarFree Gold 100Tab                 79
SugarFree Gold 300Tab                 71
Neem Toothpaste 100g                  55
TIGER BALM 10G WHITE                  23
Nivea Shaving Foam Original 200ml      4
Name: count, dtype: int64


In [9]:
health_wellness_map = {
    'Strepsils 8p':                    'PERSONAL CARE',
    'Neem Toothpaste 100g':            'PERSONAL CARE',
    'TIGER BALM 10G WHITE':            'PERSONAL CARE',
    'Nivea Shaving Foam Original 200ml': 'PERSONAL CARE',
    'SugarFree Gold 500Tab':           'FOOD STAPLES',
    'SugarFree Gold 100Tab':           'FOOD STAPLES',
    'SugarFree Gold 300Tab':           'FOOD STAPLES',
}

for product, new_category in health_wellness_map.items():
    df.loc[df['product'] == product, 'category'] = new_category

print("HEALTH AND WELLNESS products moved!")
print(f"Remaining HEALTH AND WELLNESS rows: {(df['category'] == 'HEALTH AND WELLNESS').sum()}")
print(f"Total categories now: {df['category'].nunique()}")

HEALTH AND WELLNESS products moved!
Remaining HEALTH AND WELLNESS rows: 0
Total categories now: 25


### Why we map to 25 categories
The store has 38 raw product groups but many are duplicates or near-duplicates. For example SOAP AND CLEANER and CLEANING SUPPLIES are the same thing. We merge them into 25 clean standard categories so the market basket analysis works at a meaningful level. PET CARE is dropped because it has only 1 row and cannot produce any useful patterns.

## Step 6: Remove Duplicate Rows

In [10]:
before = len(df)
df = df.drop_duplicates()
after = len(df)

print(f"Rows before: {before:,}")
print(f"Rows after:  {after:,}")
print(f"Duplicates removed: {before - after:,}")

Rows before: 768,221
Rows after:  767,181
Duplicates removed: 1,040


### Key Finding: duplicates removed
1,040 duplicate rows removed. This matches exactly what the data audit in notebook 01 found, which confirms our audit was accurate.

## Step 7: Remove Blank and Invalid Rows

In [11]:
before = len(df)
df = df.dropna(subset=['product_group'])
after = len(df)

print(f"Rows before: {before:,}")
print(f"Rows after:  {after:,}")
print(f"Rows removed: {before - after:,}")

Rows before: 767,181
Rows after:  767,180
Rows removed: 1


## Step 8: Final Verification

In [12]:
print("FINAL DATASET VERIFICATION")
print("-" * 50)
print(f"Total rows:          {len(df):,}")
print(f"Total columns:       {len(df.columns)}")
print(f"Total categories:    {df['category'].nunique()}")
print(f"Total invoices:      {df['invoice_no'].nunique():,}")
print(f"Total products:      {df['product'].nunique():,}")
print(f"Duplicate rows:      {df.duplicated().sum():,}")
print(f"Missing values:      {df.isnull().sum().sum():,}")
print(f"\nColumns: {df.columns.tolist()}")

FINAL DATASET VERIFICATION
--------------------------------------------------
Total rows:          767,180
Total columns:       14
Total categories:    25
Total invoices:      218,037
Total products:      5,680
Duplicate rows:      0
Missing values:      157

Columns: ['date', 'date_nepali', 'invoice_no', 'customer', 'product_group', 'product', 'unit', 'quantity', 'unit_price', 'base_amount', 'vat', 'total_amount', 'product_group_clean', 'category']


## Step 9: Save Cleaned Data

In [14]:
import os

output_path = r'D:\softwarica\Sem 6\Individual Project\product-placement-optimization\data\processed\sales_data_cleaned.csv'

os.makedirs(os.path.dirname(output_path), exist_ok=True)

df.to_csv(output_path, index=False)

print("Cleaned data saved successfully!")
print(f"Location: {output_path}")
print(f"Final shape: {df.shape}")

Cleaned data saved successfully!
Location: D:\softwarica\Sem 6\Individual Project\product-placement-optimization\data\processed\sales_data_cleaned.csv
Final shape: (767180, 14)


## Cleaning Complete!

### Summary of all cleaning steps performed:

1. Loaded raw data: 768,222 rows, 14 columns
2. Renamed all 14 columns to clean lowercase names
3. Dropped DISCOUNT column - zero positive values across all rows
4. Dropped ROUND OFF column - maximum value Rs 0.50, no analytical value
5. Created product_group_clean - stripped spaces, uppercase, removed underscores
6. Created category column - standardized 37 groups to 25 clean categories
7. Moved HEALTH AND WELLNESS products to correct categories
8. Dropped PET CARE - only 1 row, no analytical value
9. Removed 1,040 duplicate rows
10. Removed 1 blank row with missing product group

### Final dataset:
- Rows: 767,180
- Columns: 14
- Categories: 25
- Duplicates: 0
- Saved to: data/processed/sales_data_cleaned.csv

### Next step: 03_eda.ipynb